In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except:
    pass

In [ ]:
import os

drive_folder_name = "ProductVoiceAnalytics"
os.chdir(f'/content/drive/MyDrive/Projects/GitHub/{drive_folder_name}')

!pwd

In [ ]:
import os

# Move to repo root if running from notebooks/
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

print(os.getcwd())  # confirm it's now repo root

# 01 — Exploratory Data Analysis

Goal: Understand the raw Amazon Electronics reviews dataset before any modeling.

Key questions:
- What does the rating distribution look like?
- Are reviews balanced across classes?
- What is the typical review length?
- Are there nulls or quality issues to handle?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import RAW_REVIEWS_PATH

# Load 500K rows in chunks to avoid memory issues — full file is 3GB
df = pd.read_json(RAW_REVIEWS_PATH, lines=True, nrows=500_000)

print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")

## Basic Overview

In [ ]:
# Show first 3 rows to understand the structure
df.head(3)

In [ ]:
# Check for nulls — we only care about reviewText and overall
null_counts = df[['reviewText', 'overall']].isnull().sum()
print(null_counts)

## Rating Distribution

Expecting heavy skew toward 5-star reviews — this is typical of Amazon data.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot — how many reviews per star rating
rating_counts = df['overall'].value_counts().sort_index()
rating_counts.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Rating Distribution (Count)')
axes[0].set_xlabel('Star Rating')
axes[0].set_ylabel('Count')

# Percentage plot — same but normalized
rating_pct = df['overall'].value_counts(normalize=True).sort_index() * 100
rating_pct.plot(kind='bar', ax=axes[1], color='salmon', edgecolor='black')
axes[1].set_title('Rating Distribution (%)')
axes[1].set_xlabel('Star Rating')
axes[1].set_ylabel('Percentage')

# plt.tight_layout()
plt.show()

## Sentiment Label Distribution

Mapping: 1-2 stars → negative, 3 stars → neutral, 4-5 stars → positive

In [ ]:
def map_sentiment(rating):
    # hint: three conditions — negative, neutral, positive
    is_negative = rating <= 2
    is_neutral  = rating == 3
    is_positive = rating >= 4

    if is_negative: return 'negative'
    elif is_neutral: return 'neutral'
    else: return 'positive'

df['sentiment'] = df['overall'].apply(map_sentiment)
sentiment_stats = df['sentiment'].value_counts()
print(sentiment_stats)

# How imbalanced is it? max count / min count
imbalance_ratio = sentiment_stats.max() / sentiment_stats.min()
print(f"\nClass imbalance ratio: {imbalance_ratio:.1f}x")

## Review Length Analysis

In [ ]:
# Word count per review — split on whitespace
df['review_length'] = df["reviewText"].dropna().apply(lambda x: len(x.split()))

print(df['review_length'].describe())

# Plot distribution, clip at 500 words so outliers don't squash the chart
df['review_length'].clip(upper=300).hist(bins=50, figsize=(10, 4), color='steelblue', edgecolor='black')
plt.title('Review Length Distribution (words, clipped at 500)')
plt.xlabel('Word Count')
plt.ylabel('Frequency')
plt.show()

## Review Length by Sentiment

In [ ]:
# Drop rows where reviewText is null before plotting
df_valid = df.dropna(subset=["reviewText"])

fig, ax = plt.subplots(figsize=(8, 4))

# KDE plot per sentiment group
for sentiment, group in df_valid.groupby("sentiment"):
    group['review_length'].clip(upper=300).plot(kind='kde', ax=ax, label=sentiment)

ax.set_title('Review Length by Sentiment')
ax.set_xlabel('Word Count')
ax.legend()
plt.show()

## Verified Purchase Distribution

In [ ]:
if 'verified' in df.columns:
    # verified_counts = df["verified"].sum()
    verified_counts = df['verified'].value_counts()
    verified_pct = df["verified"].mean() * 100
    print(verified_counts)
    print(f"\n{verified_pct:.1f}% of reviews are verified purchases")

## Key Takeaways

- [ ] Note class imbalance and whether stratified sampling is needed
- [ ] Note typical review length (informs max_length for tokenizer later)
- [ ] Note null rate in reviewText
- [ ] Note any other data quality issues observed

In [ ]:
# Save a small clean sample for quick iteration in later notebooks
sample = df.dropna(subset=["reviewText", "overall"]).sample(n=10_000, random_state=42)
sample.to_csv('data/processed/eda_sample_10k.csv', index=False)
print("Saved eda_sample_10k.csv")